# Step 1: Install the required packages/libraries 

In [ ]:
# Step 1: Install the required packages/libraries 

# pip install pandas
# pip install openpyxl

# Step 2: Import required libraries

In [2]:
import sqlite3
import pandas as pd


# Step 3: Load/Import data & Build connection

In [3]:
# Load Excel file
df = pd.read_excel(r"customer_sample_500.csv.xlsx", dtype=str)

# Replace NaN with empty string
df = df.fillna("")

# Connect to SQLite
conn = sqlite3.connect(":memory:")

# Load dataframe into SQLite table
df.to_sql("customers", conn, index=False, if_exists="replace")

# Create cursor
cursor = conn.cursor()

print("Data successfully loaded into SQLite table: customers")

Data successfully loaded into SQLite table: customers


# Step 4: Data Quality | Completeness Check

## Datasets using: 
- customer_sample_500


In [21]:
import pandas as pd
import os

#-----Completeness check------

def completeness_check(cursor, df, output_file="QualityCheck_results.xlsx"):

    results = []

    # Loop through each column
    for col in df.columns:

        query = f"""
        SELECT COUNT(*)
        FROM customers
        WHERE {col} IS NULL OR TRIM({col}) = ''
        """

        # Execute query
        missing = cursor.execute(query).fetchone()[0]

        results.append({
            "Column": col,
            "Missing Values": missing
        })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    # Folder path
    folder_path = "output_files"

    # Create folder if it doesn't exist
    os.makedirs(folder_path, exist_ok=True)

    # Full file path
    file_path = os.path.join(folder_path, output_file)


    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:

        result_df.to_excel(writer, sheet_name="Completeness check", index=False)

    print("Output saved to", file_path)

    return result_df


# Run completeness check
result = completeness_check(cursor, df)

Output saved to output_files\QualityCheck_results.xlsx


# Step -5: Data Quality | Uniquenss check
## Datasets using:
- customer_sample_500

In [23]:
import pandas as pd
import os

'''Uniqueness_check'''

# -------- Step 1: Detect columns that contain only unique values --------

def find_unique_columns_sql(cursor, df):

    unique_columns = []

    for col in df.columns:

        query = f"""
        SELECT COUNT(*) = COUNT(DISTINCT {col})
        FROM customers
        """

        result = cursor.execute(query).fetchone()[0]

        if result == 1:
            unique_columns.append(col)

    return unique_columns


# Run Step 1
unique_cols = find_unique_columns_sql(cursor, df)

# Convert to dataframe
unique_df = pd.DataFrame(unique_cols, columns=["Unique_Columns"])


# -------- Step 2: Check duplicate count --------

duplicate_results = []

for col in unique_cols:

    query = f"""
    SELECT COUNT(*) - COUNT(DISTINCT {col})
    FROM customers
    """

    duplicates = cursor.execute(query).fetchone()[0]

    duplicate_results.append({
        "Column": col,
        "Duplicate_Count": duplicates
    })

duplicate_df = pd.DataFrame(duplicate_results)


# -------- Save both outputs in SAME Excel file but DIFFERENT sheets --------

file_path = "output_files/QualityCheck_results.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:

    unique_df.to_excel(writer, sheet_name="Unique Columns", index=False)
    duplicate_df.to_excel(writer, sheet_name="Duplicate Check", index=False)


print("Uniqueness results added to QualityCheck_results.xlsx in new sheets")

Uniqueness results added to QualityCheck_results.xlsx in new sheets


# Step-6 : Quality check | Validity check
## Datasets using :
- customer_sample_500 

In [24]:
#------ Validity_check------

def run_validity_checks(cursor):

    validity_rules = {

        "Invalid_SIRET": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(SIRET) != 14
        """,

        "Invalid_SIREN": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(SIREN) != 9
        """,

        "Invalid_Country_Code": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(COUNTRY_CD) != 2
        """,

        "Invalid_SAP_Postal_Code": """
            SELECT COUNT(*) FROM customers
            WHERE SAP_POSTAL_ZIP_CD GLOB '*[^0-9]*'
        """,

        "Invalid_LUCI_Postal_Code": """
            SELECT COUNT(*) FROM customers
            WHERE LUCI_POSTAL_ZIP_CD GLOB '*[^0-9]*'
        """,

        "Invalid_Customer_Delivery_Block": """
            SELECT COUNT(*) FROM customers
            WHERE CUSTOMER_DELIVERY_BLOC NOT IN ('00','01')
        """,

        "Invalid_GERS": """
            SELECT COUNT(*) FROM customers
            WHERE GERS GLOB '*[^0-9]*'
        """
    }

    delivery_columns = [
        "CUSTOMER_ORDER_BLOCK",
        "CUSTOMER_SALES_DELIVERY_BLOC",
        "CUSTOMER_SALES_ORDER_BLOCK",
        "DELETION_BLOCK"
    ]

    for col in delivery_columns:
        validity_rules[f"{col}_Invalid"] = f"""
            SELECT COUNT(*) FROM customers
            WHERE {col} NOT IN ('00','01')
        """

    results = []

    for rule, query in validity_rules.items():
        invalid_count = cursor.execute(query).fetchone()[0]

        results.append({
            "Check": rule,
            "Invalid_Count": invalid_count
        })

    validity_df = pd.DataFrame(results)

    # existing excel file path
    file_path = "output_files/QualityCheck_results.xlsx"

    # append new sheet
    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        validity_df.to_excel(writer, sheet_name="Validity_Check", index=False)

    print("Validity results added to QualityCheck_results.xlsx")


# run validity checks
run_validity_checks(cursor)

Validity results added to QualityCheck_results.xlsx


# Step-7: Quality Checks | Consistency check
## Data sets using :
- customer_sample_500

In [12]:
import pandas as pd

#------ Consistency check -------

def run_consistency_checks(cursor):

    consistency_rules = {
        "Country_Mismatch": """
            SELECT COUNT(*)
            FROM customers
            WHERE COUNTRY_CD != LUCI_COUNTRY_CD
        """
    }

    results = []

    # Excel output location
    file_path = "output_files/QualityCheck_results.xlsx"

    for rule, query in consistency_rules.items():
        mismatch_count = cursor.execute(query).fetchone()[0]

        results.append({
            "Check": rule,
            "Mismatch_Count": mismatch_count,
            "Output_Location": file_path
        })

    consistency_df = pd.DataFrame(results)

    # Append new sheet
    with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        consistency_df.to_excel(writer, sheet_name="Consistency_Check", index=False)

    print(f"Consistency results saved at: {file_path}")


# run consistency check
run_consistency_checks(cursor)

Consistency results saved at: output_files/QualityCheck_results.xlsx
